In [14]:
import requests
from bs4 import BeautifulSoup
import json
import time
import csv
import re
import os
from datetime import datetime
from typing import Dict, List, Optional
import random
from urllib.parse import urljoin, urlparse

class YandexRealtyScraperLight:

    def __init__(self, download_images=False):
        self.download_images = download_images
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
            'Accept-Language': 'ru-RU,ru;q=0.9,en-US;q=0.8,en;q=0.7',
            'Accept-Encoding': 'gzip, deflate, br',
            'Connection': 'keep-alive',
            'Upgrade-Insecure-Requests': '1',
        })
        
    def _make_request(self, url: str, max_retries: int = 3) -> Optional[requests.Response]:

        for attempt in range(max_retries):
            try:
                time.sleep(random.uniform(1, 2))
                response = self.session.get(url, timeout=30)
                response.raise_for_status()
                return response
            except Exception as e:
                if attempt == max_retries - 1:
                    print(f"Ошибка при запросе {url}: {e}")
                    return None
                time.sleep(2 ** attempt)
        return None
    
    def get_listings(self, city: str = "moskva", pages: int = 2, max_offers: int = 100) -> List[str]:
        listing_urls = []
        
        for page in range(1, pages + 1):
            if len(listing_urls) >= max_offers:
                break
            
            try:
                if page == 1:
                    url = f"https://realty.yandex.ru/{city}/kupit/kvartira/"
                else:
                    url = f"https://realty.yandex.ru/{city}/kupit/kvartira/?page={page}"
                
                print(f"📄 Загружаем страницу {page}: {url}")
                response = self._make_request(url)
                
                if not response:
                    continue
                
                soup = BeautifulSoup(response.content, 'html.parser')

                links = []

                cards = soup.select('[data-testid="offer-card"]')
                for card in cards:
                    link = card.find('a')
                    if link and link.get('href'):
                        href = link.get('href')
                        if '/offer/' in href:
                            full_url = urljoin('https://realty.yandex.ru', href)
                            links.append(full_url)

                if not links:
                    for a in soup.find_all('a', href=True):
                        if '/offer/' in a['href']:
                            full_url = urljoin('https://realty.yandex.ru', a['href'])
                            if full_url not in links:
                                links.append(full_url)
                
                if links:
                    listing_urls.extend(links)
                    print(f"    Найдено {len(links)} объявлений на странице {page}")
                else:
                    print(f"    Не найдено объявлений на странице {page}")
                
                time.sleep(random.uniform(1, 2))
            except Exception as e:
                print(f" Ошибка при загрузке страницы {page}: {e}")
                continue
        
        listing_urls = list(set(listing_urls))[:max_offers]
        print(f"\n Всего найдено уникальных объявлений: {len(listing_urls)}")
        return listing_urls
    
    def parse_structured_data(self, soup: BeautifulSoup) -> Dict:

        data = {}

        scripts = soup.find_all('script', type='application/ld+json')
        for script in scripts:
            try:
                json_data = json.loads(script.string)
                if isinstance(json_data, dict):

                    if 'offers' in json_data:
                        if isinstance(json_data['offers'], dict):
                            data['price'] = json_data['offers'].get('price')
                        elif isinstance(json_data['offers'], list) and json_data['offers']:
                            data['price'] = json_data['offers'][0].get('price')
                    
                    if 'geo' in json_data:
                        data['latitude'] = json_data['geo'].get('latitude')
                        data['longitude'] = json_data['geo'].get('longitude')
              
                    if 'description' in json_data:
                        data['json_description'] = json_data['description']
            except:
                continue
        
        if not data.get('price'):
            price_elem = soup.find('span', {'data-testid': 'price-value'})
            if price_elem:
                price_text = price_elem.get_text(strip=True)
                price_match = re.search(r'[\d\s]+', price_text)
                if price_match:
                    data['price'] = int(re.sub(r'[^\d]', '', price_match.group()))
        
        area_elem = soup.find('span', {'data-testid': 'total-area'})
        if area_elem:
            area_text = area_elem.get_text(strip=True)
            area_match = re.search(r'[\d.,]+', area_text)
            if area_match:
                try:
                    data['total_area'] = float(area_match.group().replace(',', '.'))
                except:
                    pass
        
        floor_elem = soup.find('span', {'data-testid': 'floor'})
        if floor_elem:
            floor_text = floor_elem.get_text(strip=True)
            floors = re.findall(r'\d+', floor_text)
            if floors:
                data['floor'] = int(floors[0])
            if len(floors) > 1:
                data['total_floors'] = int(floors[1])
        
        rooms_elem = soup.find('span', {'data-testid': 'rooms'})
        if rooms_elem:
            rooms_text = rooms_elem.get_text(strip=True).lower()
            if 'студия' in rooms_text:
                data['rooms'] = 'студия'
            else:
                rooms_match = re.search(r'\d+', rooms_text)
                if rooms_match:
                    data['rooms'] = int(rooms_match.group())
        
        address_elem = soup.find('span', {'data-testid': 'address'})
        if address_elem:
            data['address'] = address_elem.get_text(strip=True)
        
        return data
    
    def parse_description(self, soup: BeautifulSoup) -> str:
        selectors = [
            ('div', {'data-testid': 'offer-description'}),
            ('div', {'class': 'OfferDescription'}),
            ('div', {'itemprop': 'description'}),
        ]
        
        for tag, attrs in selectors:
            elem = soup.find(tag, attrs)
            if elem:
                text = elem.get_text(strip=True)
                if text:
                    return text
        
        return ""
    
    def parse_images(self, soup: BeautifulSoup) -> List[str]:
        images = []
        
        scripts = soup.find_all('script', type='application/ld+json')
        for script in scripts:
            try:
                json_data = json.loads(script.string)
                if 'image' in json_data:
                    if isinstance(json_data['image'], list):
                        images.extend(json_data['image'])
                    elif isinstance(json_data['image'], str):
                        images.append(json_data['image'])
            except:
                continue
        
        gallery = soup.find('div', {'data-testid': 'gallery'})
        if gallery:
            for img in gallery.find_all('img'):
                src = img.get('src') or img.get('data-src')
                if src and src.startswith('http') and 'blob:' not in src:
                    src = src.split('?')[0]
                    if src not in images:
                        images.append(src)
        
        images = list(dict.fromkeys(images))
        
        return images[:20]
    
    def scrape_listing(self, url: str) -> Optional[Dict]:
            response = self._make_request(url)
            if not response:
                return None
            
            soup = BeautifulSoup(response.content, 'html.parser')
            
            structured = self.parse_structured_data(soup)
            
            data = {
                'url': url,
                'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                'title': self._get_title(soup),
                'description': self.parse_description(soup),
                'images': self.parse_images(soup),
                **structured
            }
            
            return data

    
    def _get_title(self, soup: BeautifulSoup) -> str:
        title_elem = soup.find('h1')
        if title_elem:
            return title_elem.get_text(strip=True)
        return ""
    
    def save_to_csv(self, data: List[Dict], filename: str = "realty_data_text_and_numbers.csv"):
        if not data:
            print(" Нет данных для сохранения")
            return
        
        fieldnames = [
            'timestamp', 'title', 'price', 'total_area', 'rooms',
            'floor', 'total_floors', 'address', 'latitude', 'longitude',
            'description', 'json_description', 'url', 'images_count'
        ]
        
        with open(filename, 'w', newline='', encoding='utf-8-sig') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
            writer.writeheader()
            
            for item in data:
                row = {}
                for field in fieldnames:
                    value = item.get(field, '')
                    if value is None:
                        value = ''
                    if field == 'images_count':
                        row[field] = len(item.get('images', []))
                    else:
                        row[field] = value
                writer.writerow(row)
        
        print(f"\n Данные сохранены в {filename}")
        print(f"   Всего записей: {len(data)}")
    
    def print_statistics(self, data: List[Dict]):
        if not data:
            return
        
        total = len(data)
        with_desc = sum(1 for item in data if item.get('description'))
        with_images = sum(1 for item in data if item.get('images'))
        with_price = sum(1 for item in data if item.get('price'))
        

        print(f" Всего объявлений: {total}")
        print(f" С описанием: {with_desc} ({with_desc/total*100:.1f}%)")
        print(f" С изображениями: {with_images} ({with_images/total*100:.1f}%)")
        print(f" С указанием цены: {with_price} ({with_price/total*100:.1f}%)")
        
        prices = [item['price'] for item in data if item.get('price')]
        if prices:
            avg_price = sum(prices) / len(prices)
            print(f"\n Цены:")
            print(f"   Средняя: {avg_price:,.0f} ")
            print(f"   Минимальная: {min(prices):,.0f} ")
            print(f"   Максимальная: {max(prices):,.0f}")
        
        areas = [item['total_area'] for item in data if item.get('total_area')]
        if areas:
            print(f"\n Площади:")
            print(f"   Средняя: {sum(areas)/len(areas):.1f}")
            print(f"   Минимальная: {min(areas):.1f}")
            print(f"   Максимальная: {max(areas):.1f}")


def main():

    CITY = "moskva"
    PAGES = 131
    MAX_OFFERS = 3000
    
    scraper = YandexRealtyScraperLight(download_images=False)
    
    try:

        listing_urls = scraper.get_listings(
            city=CITY, 
            pages=PAGES, 
            max_offers=MAX_OFFERS
        )
        
        if not listing_urls:
            print(" Не найдено объявлений.")
            return

        scraped_data = []
        
        for i, url in enumerate(listing_urls, 1):
            print(f"\n[{i}/{len(listing_urls)}]", end=" ")
            data = scraper.scrape_listing(url)
            
            if data:
                scraped_data.append(data)
                price = data.get('price', 'N/A')
                if price != 'N/A':
                    price = f"{price:,} ₽"
                area = data.get('total_area', 'N/A')
                print(f"    Цена: {price} | Площадь: {area} м2 | Изображений: {len(data.get('images', []))}")
            else:
                print(f"    Не удалось собрать данные")

        if scraped_data:
            scraper.save_to_csv(scraped_data)
            scraper.print_statistics(scraped_data)
            
        else:
            print(" Не удалось собрать данные")
            
    except KeyboardInterrupt:
        print("\n\n Прервано пользователем")
    
if __name__ == "__main__":
    main()

📄 Загружаем страницу 1: https://realty.yandex.ru/moskva/kupit/kvartira/
    Найдено 23 объявлений на странице 1
📄 Загружаем страницу 2: https://realty.yandex.ru/moskva/kupit/kvartira/?page=2
    Найдено 23 объявлений на странице 2
📄 Загружаем страницу 3: https://realty.yandex.ru/moskva/kupit/kvartira/?page=3
    Найдено 23 объявлений на странице 3
📄 Загружаем страницу 4: https://realty.yandex.ru/moskva/kupit/kvartira/?page=4
    Найдено 23 объявлений на странице 4
📄 Загружаем страницу 5: https://realty.yandex.ru/moskva/kupit/kvartira/?page=5
    Найдено 23 объявлений на странице 5
📄 Загружаем страницу 6: https://realty.yandex.ru/moskva/kupit/kvartira/?page=6
    Найдено 23 объявлений на странице 6
📄 Загружаем страницу 7: https://realty.yandex.ru/moskva/kupit/kvartira/?page=7
    Найдено 23 объявлений на странице 7
📄 Загружаем страницу 8: https://realty.yandex.ru/moskva/kupit/kvartira/?page=8
    Найдено 23 объявлений на странице 8
📄 Загружаем страницу 9: https://realty.yandex.ru/moskva